# Tahap 4 — Daily Sounding Selection

**Tujuan notebook**
Membangun dataset sounding harian resmi (`03_daily_sounding_selection.csv`) yang menjadi
input Tahap 5 (Integrasi Dataset), berdasarkan hasil validasi kalender Tahap 3.

**Konteks penting (hasil Tahap 3):**
Audit Tahap 3 menunjukkan tidak ada tanggal dengan kombinasi `00Z,12Z` — setiap tanggal
paling banyak memiliki satu sounding. Karena itu Tahap 4 di sini **bukan** melakukan
pemilihan antara 12Z dan 00Z (tidak ada konflik untuk dipilih), melainkan **memformalkan**
jam yang sudah tersedia menjadi kolom `selected_hour` dan `selection_status` yang siap
dipakai tahap berikutnya.

**Batasan cakupan (scope) — Tahap 4 TIDAK mencakup:**
- merge dengan Ogimet
- imputasi
- interpolasi
- labeling
- outlier handling
- feature engineering

**Deliverable:**
- `03_daily_sounding_selection.csv` (kolom: `date`, `selected_hour`, `selection_status`)
- `STAGE4_REPORT.md`


## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


## 2. Konfigurasi

In [2]:
INPUT_DIR = Path(".")
CALENDAR_VALIDATION_PATH = INPUT_DIR / "02_calendar_validation.csv"
SOUNDERPY_PATH = INPUT_DIR / "sounderpy_standardized.csv"

OUTPUT_CSV_PATH = Path("03_daily_sounding_selection.csv")
OUTPUT_REPORT_PATH = Path("STAGE4_REPORT.md")

# jam_tersedia -> (selected_hour, selection_status)
SELECTION_RULE = {
    "12Z": ("12Z", "SELECTED"),
    "00Z": ("00Z", "SELECTED"),
    "NONE": ("MISSING", "NO_SOUNDING"),
}


## 3. Langkah 1: Load Data

Deliverable Tahap 3 (`02_calendar_validation.csv`) sudah memuat master calendar lengkap
beserta `jam_tersedia` per tanggal — ini adalah sumber utama untuk pemetaan seleksi.
`sounderpy_standardized.csv` dibaca hanya untuk keperluan validasi konsistensi silang
(bukan untuk membentuk ulang dataset), sehingga hanya kolom `date` dan `hour` yang
diambil.

In [3]:
def load_calendar_validation(path: Path) -> pd.DataFrame:
    """Load hasil Tahap 3 sebagai dasar seleksi sounding harian."""
    df = pd.read_csv(path)
    df["tanggal"] = pd.to_datetime(df["tanggal"], errors="raise")
    return df


def load_sounderpy_raw(path: Path) -> pd.DataFrame:
    """Load hanya kolom date dan hour dari SounderPy mentah, untuk validasi silang."""
    df = pd.read_csv(path, usecols=["date", "hour"])
    df["date"] = pd.to_datetime(df["date"], errors="raise")
    return df


calendar_validation = load_calendar_validation(CALENDAR_VALIDATION_PATH)
sounderpy_raw = load_sounderpy_raw(SOUNDERPY_PATH)

calendar_validation.head()


,tanggal,status_ogimet,status_sounderpy,jam_tersedia,kategori
0,2017-01-01,ADA,ADA,12Z,BOTH
1,2017-01-02,ADA,ADA,12Z,BOTH
2,2017-01-03,ADA,ADA,12Z,BOTH
3,2017-01-04,ADA,ADA,12Z,BOTH
4,2017-01-05,ADA,ADA,12Z,BOTH


## 4. Langkah 2: Terapkan Aturan Seleksi

Aturan diterapkan langsung dari kolom `jam_tersedia` (bukan dari raw SounderPy), karena
`jam_tersedia` sudah merupakan hasil audit resmi Tahap 3 per tanggal kalender. Seluruh
tanggal kalender dipertahankan (`date` sebagai primary key, 2017-01-01 s.d. 2024-12-31).

Aturan hanya mengenali tiga nilai `jam_tersedia`: `12Z`, `00Z`, `NONE`. Nilai lain
(mis. `00Z,12Z`) akan memicu error eksplisit agar tidak diselesaikan secara diam-diam,
karena Tahap 3 sudah mengonfirmasi tidak ada kasus tersebut pada dataset ini.

In [4]:
def apply_selection_rule(df: pd.DataFrame) -> pd.DataFrame:
    """Petakan jam_tersedia menjadi selected_hour dan selection_status."""
    unknown = sorted(set(df["jam_tersedia"].unique()) - set(SELECTION_RULE.keys()))
    if unknown:
        raise ValueError(
            f"Nilai jam_tersedia tidak dikenali oleh aturan Tahap 4: {unknown}. "
            "Perlu peninjauan manual sebelum melanjutkan."
        )

    result = df[["tanggal"]].rename(columns={"tanggal": "date"}).copy()
    mapped = df["jam_tersedia"].map(SELECTION_RULE)
    result["selected_hour"] = mapped.map(lambda t: t[0])
    result["selection_status"] = mapped.map(lambda t: t[1])
    return result


daily_sounding_selection = apply_selection_rule(calendar_validation)
daily_sounding_selection.head()


,date,selected_hour,selection_status
0,2017-01-01,12Z,SELECTED
1,2017-01-02,12Z,SELECTED
2,2017-01-03,12Z,SELECTED
3,2017-01-04,12Z,SELECTED
4,2017-01-05,12Z,SELECTED


## 5. Validasi Konsistensi

Dua lapis validasi:
1. **Integritas struktural** — jumlah baris sesuai master calendar, tidak ada duplicate
   date, seluruh tanggal kalender dipertahankan.
2. **Validasi silang terhadap SounderPy mentah** — untuk tanggal `SELECTED`, jam yang
   dipilih memang benar-benar ada pada data mentah; untuk tanggal `NO_SOUNDING`, tanggal
   tersebut memang tidak muncul sama sekali pada data mentah.

In [5]:
def validate_structure(df: pd.DataFrame, expected_days: int) -> dict:
    """Validasi integritas struktural dataset seleksi sounding harian."""
    n_rows = len(df)
    n_duplicate_dates = int(df["date"].duplicated().sum())
    n_unique_dates = df["date"].nunique()

    checks = {
        "total_row_sesuai_master_calendar": n_rows == expected_days,
        "tidak_ada_duplicate_date": n_duplicate_dates == 0,
        "semua_tanggal_unik_dipertahankan": n_unique_dates == expected_days,
    }
    return {
        "n_rows": n_rows,
        "n_duplicate_dates": n_duplicate_dates,
        "n_unique_dates": n_unique_dates,
        "checks": checks,
    }


def validate_cross_source(selection_df: pd.DataFrame, sounderpy_raw: pd.DataFrame) -> dict:
    """Validasi silang selected_hour terhadap keberadaan aktual di SounderPy mentah."""
    hour_label = {0: "00Z", 12: "12Z"}
    raw_hours_by_date = sounderpy_raw.groupby("date")["hour"].apply(
        lambda s: set(hour_label.get(h, f"{h:02d}Z") for h in s)
    )

    mismatches = []
    for row in selection_df.itertuples(index=False):
        raw_hours = raw_hours_by_date.get(row.date, set())
        if row.selection_status == "SELECTED":
            if row.selected_hour not in raw_hours:
                mismatches.append((row.date, "selected_hour_not_in_raw"))
        else:  # NO_SOUNDING
            if len(raw_hours) > 0:
                mismatches.append((row.date, "raw_has_sounding_but_marked_no_sounding"))

    return {
        "n_mismatch": len(mismatches),
        "mismatch_examples": mismatches[:5],
    }


structural_result = validate_structure(daily_sounding_selection, expected_days=len(calendar_validation))
cross_source_result = validate_cross_source(daily_sounding_selection, sounderpy_raw)

structural_result, cross_source_result


({'n_rows': 2922,
  'n_duplicate_dates': 0,
  'n_unique_dates': 2922,
  'checks': {'total_row_sesuai_master_calendar': True,
   'tidak_ada_duplicate_date': True,
   'semua_tanggal_unik_dipertahankan': True}},
 {'n_mismatch': 0, 'mismatch_examples': []})

In [6]:
all_checks_passed = all(structural_result["checks"].values()) and cross_source_result["n_mismatch"] == 0
assert all_checks_passed, "Validasi konsistensi Tahap 4 gagal — periksa hasil di atas sebelum lanjut."
print("Seluruh validasi konsistensi LULUS.")


Seluruh validasi konsistensi LULUS.


## 6. Simpan Output CSV

In [7]:
output_columns = ["date", "selected_hour", "selection_status"]
daily_sounding_selection[output_columns].to_csv(OUTPUT_CSV_PATH, index=False)

print(f"Tersimpan: {OUTPUT_CSV_PATH.resolve()}")


Tersimpan: /home/claude/work/03_daily_sounding_selection.csv


## 7. Audit Ringkas

In [8]:
def compute_audit(df: pd.DataFrame) -> dict:
    """Hitung ringkasan audit Tahap 4."""
    total_hari = len(df)
    n_selected = int((df["selection_status"] == "SELECTED").sum())
    n_no_sounding = int((df["selection_status"] == "NO_SOUNDING").sum())
    distribusi_selected_hour = df["selected_hour"].value_counts().to_dict()

    return {
        "total_hari": total_hari,
        "n_selected": n_selected,
        "n_no_sounding": n_no_sounding,
        "distribusi_selected_hour": distribusi_selected_hour,
    }


audit_result = compute_audit(daily_sounding_selection)

print(f"Total hari       : {audit_result['total_hari']}")
print(f"Jumlah SELECTED  : {audit_result['n_selected']}")
print(f"Jumlah NO_SOUNDING: {audit_result['n_no_sounding']}")
print("Distribusi selected_hour:")
for jam, jumlah in sorted(audit_result["distribusi_selected_hour"].items()):
    print(f"  {jam}: {jumlah}")


Total hari       : 2922
Jumlah SELECTED  : 2774
Jumlah NO_SOUNDING: 148
Distribusi selected_hour:
  00Z: 460
  12Z: 2314
  MISSING: 148


## 8. Cek Acceptance Criteria

In [9]:
def check_acceptance_criteria(audit: dict, structural: dict) -> pd.DataFrame:
    """Bandingkan hasil aktual terhadap acceptance criteria Tahap 4."""
    rows = [
        ("Total row = 2922", audit["total_hari"] == 2922, audit["total_hari"]),
        ("SELECTED = 2774", audit["n_selected"] == 2774, audit["n_selected"]),
        ("NO_SOUNDING = 148", audit["n_no_sounding"] == 148, audit["n_no_sounding"]),
        ("Tidak ada duplicate date", structural["n_duplicate_dates"] == 0, structural["n_duplicate_dates"]),
        (
            "Semua tanggal kalender dipertahankan",
            structural["n_unique_dates"] == structural["n_rows"],
            structural["n_unique_dates"],
        ),
    ]
    return pd.DataFrame(rows, columns=["kriteria", "terpenuhi", "nilai_aktual"])


acceptance_df = check_acceptance_criteria(audit_result, structural_result)
acceptance_df


,kriteria,terpenuhi,nilai_aktual
0,Total row = 2922,True,2922
1,SELECTED = 2774,True,2774
2,NO_SOUNDING = 148,True,148
3,Tidak ada duplicate date,True,0
4,Semua tanggal kalender dipertahankan,True,2922


## 9. Susun STAGE4_REPORT.md

In [10]:
def build_stage4_report(audit: dict, structural: dict, cross_source: dict, acceptance_df: pd.DataFrame) -> str:
    """Bangun konten STAGE4_REPORT.md dari hasil audit dan validasi aktual."""
    dist_lines = "\n".join(
        f"- {jam}: {jumlah} hari" for jam, jumlah in sorted(audit["distribusi_selected_hour"].items())
    )

    acceptance_lines = "\n".join(
        f"- [{'x' if row.terpenuhi else ' '}] {row.kriteria} (aktual: {row.nilai_aktual})"
        for row in acceptance_df.itertuples(index=False)
    )

    status_konsistensi = "LULUS" if cross_source["n_mismatch"] == 0 else "GAGAL"

    report = f"""# STAGE4_REPORT — Daily Sounding Selection

## Ringkasan

- Jumlah record (baris) : {audit['total_hari']}
- Jumlah SELECTED        : {audit['n_selected']}
- Jumlah NO_SOUNDING      : {audit['n_no_sounding']}

## Distribusi selected_hour

{dist_lines}

## Hari Tanpa Sounding

Jumlah hari tanpa sounding (`selection_status = NO_SOUNDING`): {audit['n_no_sounding']}

## Hasil Validasi Konsistensi

- Validasi struktural (jumlah baris, duplicate date, keutuhan tanggal kalender): {'LULUS' if all(structural['checks'].values()) else 'GAGAL'}
- Validasi silang terhadap SounderPy mentah (selected_hour vs data mentah): {status_konsistensi}
- Jumlah mismatch validasi silang: {cross_source['n_mismatch']}

## Acceptance Criteria

{acceptance_lines}

## Catatan Cakupan

Tahap ini tidak melakukan merge dengan Ogimet, imputasi, interpolasi, labeling, outlier
handling, maupun feature engineering. Dataset ini adalah input resmi untuk Tahap 5
(Integrasi Dataset).
"""
    return report


report_content = build_stage4_report(audit_result, structural_result, cross_source_result, acceptance_df)
OUTPUT_REPORT_PATH.write_text(report_content)
print(f"Tersimpan: {OUTPUT_REPORT_PATH.resolve()}")
print()
print(report_content)


Tersimpan: /home/claude/work/STAGE4_REPORT.md

# STAGE4_REPORT — Daily Sounding Selection

## Ringkasan

- Jumlah record (baris) : 2922
- Jumlah SELECTED        : 2774
- Jumlah NO_SOUNDING      : 148

## Distribusi selected_hour

- 00Z: 460 hari
- 12Z: 2314 hari
- MISSING: 148 hari

## Hari Tanpa Sounding

Jumlah hari tanpa sounding (`selection_status = NO_SOUNDING`): 148

## Hasil Validasi Konsistensi

- Validasi struktural (jumlah baris, duplicate date, keutuhan tanggal kalender): LULUS
- Validasi silang terhadap SounderPy mentah (selected_hour vs data mentah): LULUS
- Jumlah mismatch validasi silang: 0

## Acceptance Criteria

- [x] Total row = 2922 (aktual: 2922)
- [x] SELECTED = 2774 (aktual: 2774)
- [x] NO_SOUNDING = 148 (aktual: 148)
- [x] Tidak ada duplicate date (aktual: 0)
- [x] Semua tanggal kalender dipertahankan (aktual: 2922)

## Catatan Cakupan

Tahap ini tidak melakukan merge dengan Ogimet, imputasi, interpolasi, labeling, outlier
handling, maupun feature engineering. D